# Apply Best Practice Analyzer fixes to a semantic model

This notebook runs the **Best Practice Analyzer (BPA)** on a Power BI / Fabric semantic model,
turns the findings into an editable **fix plan**, lets you **review and remove** anything you do
not want changed, and then **applies the approved fixes** back to the model using
[Semantic Link Labs](https://github.com/microsoft/semantic-link-labs).

**Workflow**
1. Run the BPA and see every finding.
2. Get a plan that says, for each finding, exactly what the automated fix would do.
3. Review the plan and switch `Apply` to `False` for anything you want to keep as-is.
4. Apply the approved fixes (with a dry-run first).
5. Re-run the BPA to confirm the findings are cleared.

> Safe by design: the apply step defaults to **dry run**. Nothing is written to your model
> until you explicitly set `DRY_RUN = False`.

## Step 0 - Prerequisites

- Run this in a **Microsoft Fabric notebook** (it uses the built-in Spark/Python runtime and your Fabric identity).
- You need **write/edit** permission on the target semantic model (member or admin on the workspace, or Write on the model).
- Semantic Link Labs is usually pre-installed in Fabric. If not, run the install cell below once.

In [ ]:
# Run once if sempy_labs is not already available in your Fabric runtime, then restart the kernel.
# %pip install semantic-link-labs --upgrade

## Step 1 - Configure and run the Best Practice Analyzer

Set `dataset` to your semantic model name (or ID) and `workspace` to the workspace name/ID.
Leave `workspace = None` to use the workspace this notebook lives in.

In [ ]:
import re
import pandas as pd
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

# === EDIT THESE FOR YOUR MODEL ===
dataset = "Sudsberry Property Assessment"   # semantic model name or ID
workspace = None                              # workspace name or ID; None = this notebook's workspace

In [ ]:
# Visual scorecard (Formatting / Maintenance / Performance tabs) - same as the screenshot in Fabric.
labs.run_model_bpa(dataset=dataset, workspace=workspace)

In [ ]:
# Machine-readable version of the same results, one row per finding.
bpa_df = labs.run_model_bpa(dataset=dataset, workspace=workspace, return_dataframe=True)
print(f"{len(bpa_df)} best-practice finding(s) returned.")
bpa_df

## Step 2 - Build the fix plan

The registry below maps a **BPA rule name** to an **automated remediation**. Findings whose rule
is in the registry are marked `Auto-Fixable = True` and get a `Proposed Action`. Everything else
is left for manual review (the notebook will never touch it).

**To support more rules**, add an entry to `REMEDIATIONS`. Each fix is a function
`fix(tom, row)` that uses the Semantic Link Labs TOM helpers (e.g. `tom.update_measure`,
`tom.update_column`, `tom.set_data_type`). See the commented examples at the bottom of the cell.

In [ ]:
# ---------- object-name parsers (match the format BPA returns) ----------
def _parse_column(object_name):
    """'Table'[Column] -> ('Table', 'Column')"""
    m = re.match(r"^'(.+?)'\[(.+)\]$", str(object_name))
    return (m.group(1), m.group(2)) if m else (None, None)

def _parse_relationship(object_name):
    """'T1'[C1] -> 'T2'[C2]  ->  (('T1','C1'), ('T2','C2'))"""
    try:
        left, right = str(object_name).split(" -> ")
    except ValueError:
        return (None, None), (None, None)
    return _parse_column(left.strip()), _parse_column(right.strip())

# ---------- reusable fix functions ----------
def _fix_format_string(fmt):
    def _apply(tom, row):
        ot, name = row["Object Type"], row["Object Name"]
        if ot == "Measure":
            tom.update_measure(measure_name=name, format_string=fmt)
        elif ot in ("Column", "Calculated Column"):
            t, c = _parse_column(name)
            tom.update_column(table_name=t, column_name=c, format_string=fmt)
        else:
            raise ValueError(f"Format-string fix not supported for object type '{ot}'")
        return f"Format string set to '{fmt}'"
    return _apply

def _fix_relationship_to_int(tom, row):
    (ft, fc), (tt, tc) = _parse_relationship(row["Object Name"])
    changed = []
    for tbl, col in [(ft, fc), (tt, tc)]:
        if tbl and col:
            tom.set_data_type(table_name=tbl, column_name=col, value="Int64")
            changed.append(f"'{tbl}'[{col}]")
    return "Data type set to Int64 for " + ", ".join(changed)

def _fix_hide_column(tom, row):
    t, c = _parse_column(row["Object Name"])
    tom.update_column(table_name=t, column_name=c, hidden=True)
    return "Column hidden"

def _fix_mdx_false(tom, row):
    t, c = _parse_column(row["Object Name"])
    tom.set_is_available_in_mdx(table_name=t, column_name=c, value=False)
    return "IsAvailableInMdx set to False"

def _fix_mark_key(tom, row):
    t, c = _parse_column(row["Object Name"])
    tom.update_column(table_name=t, column_name=c, key=True)
    return "Marked as key column"

# ---------- rule -> remediation registry (extend this) ----------
REMEDIATIONS = {
    "Whole numbers should be formatted with thousands separators and no decimals": {
        "action": "Set format string to '#,0'",
        "fix": _fix_format_string("#,0"),
    },
    "Percentages should be formatted with thousands separators and decimals": {
        "action": "Set format string to '#,0.00%'",
        "fix": _fix_format_string("#,0.00%"),
    },
    "Provide format string for 'Date' columns": {
        "action": "Set format string to 'mm/dd/yyyy'",
        "fix": _fix_format_string("mm/dd/yyyy"),
    },
    "Relationship columns should be of integer data type": {
        "action": "Change both relationship columns to Int64",
        "fix": _fix_relationship_to_int,
    },
    "Hide foreign keys": {
        "action": "Hide the foreign key column",
        "fix": _fix_hide_column,
    },
    "Set IsAvailableInMdx to false on non-attribute columns": {
        "action": "Set IsAvailableInMdx = False",
        "fix": _fix_mdx_false,
    },
    "Mark primary keys": {
        "action": "Mark the column as a key column",
        "fix": _fix_mark_key,
    },
}

# --- Want to auto-fix another rule? Add it here. Example: ---
# def _fix_summarize_none(tom, row):
#     t, c = _parse_column(row["Object Name"])
#     tom.set_summarize_by(table_name=t, column_name=c, value="None")
#     return "SummarizeBy set to None"
# REMEDIATIONS["Set SummarizeBy property to 'none' for non-key numeric columns"] = {
#     "action": "Set SummarizeBy = None", "fix": _fix_summarize_none }

In [ ]:
def build_plan(bpa_df):
    plan = bpa_df.copy()
    plan["Auto-Fixable"] = plan["Rule Name"].isin(REMEDIATIONS.keys())
    plan["Proposed Action"] = plan["Rule Name"].map(
        lambda r: REMEDIATIONS[r]["action"] if r in REMEDIATIONS
        else "Manual review (no automated fix registered)"
    )
    plan.insert(0, "Apply", plan["Auto-Fixable"])
    cols = ["Apply", "Auto-Fixable", "Category", "Severity", "Rule Name",
            "Object Type", "Object Name", "Proposed Action"]
    return plan[cols].reset_index(drop=True)

plan_df = build_plan(bpa_df)
print(f"Auto-fixable: {int(plan_df['Auto-Fixable'].sum())} of {len(plan_df)} finding(s). "
      f"Rows where Apply=True and Auto-Fixable=True will be changed.")
plan_df

## Step 3 - Review and trim the plan

By default every auto-fixable finding is set to `Apply = True`. Turn off anything you want to keep
unchanged. Use **either** the quick in-notebook edits below **or** the Excel round-trip.

### Option A - quick edits in the notebook

In [ ]:
# 1) Skip specific rows by their number in the left-most index of the table above:
rows_to_skip = []            # e.g. [0, 4, 7]
plan_df.loc[rows_to_skip, "Apply"] = False

# 2) Or skip by rule / category / object type (uncomment what you need):
# plan_df.loc[plan_df["Rule Name"] == "Hide foreign keys", "Apply"] = False
# plan_df.loc[plan_df["Category"] == "Performance", "Apply"] = False
# plan_df.loc[plan_df["Object Name"] == "Assessment Tax Year", "Apply"] = False

plan_df

### Option B - review in Excel (good for handing to a colleague / customer)

Requires a **Lakehouse attached** to this notebook. Writes the plan to `Files/bpa_fix_plan.csv`;
open it, set the `Apply` column to `True`/`False`, save, then run the reload cell.

In [ ]:
review_path = "/lakehouse/default/Files/bpa_fix_plan.csv"
plan_df.to_csv(review_path, index=True)
print(f"Saved to {review_path}")
print("Open it (Lakehouse > Files, or download), edit the 'Apply' column, save, then run the reload cell.")

In [ ]:
# Reload after editing the CSV:
plan_df = pd.read_csv(review_path, index_col=0)
plan_df["Apply"] = plan_df["Apply"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
plan_df["Auto-Fixable"] = plan_df["Auto-Fixable"].astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
plan_df

## Step 4 - Apply the approved fixes

First run this cell as-is (`DRY_RUN = True`) to see exactly what will change. When you are happy,
set `DRY_RUN = False` and run it again to write the changes to the model.

In [ ]:
DRY_RUN = True   # <-- set to False to actually write changes to the model

to_apply = plan_df[(plan_df["Apply"] == True) & (plan_df["Auto-Fixable"] == True)].copy()
results = []

if to_apply.empty:
    print("Nothing selected to apply (no rows with Apply=True and Auto-Fixable=True).")
elif DRY_RUN:
    print(f"DRY RUN - {len(to_apply)} change(s) would be applied:\n")
    for _, row in to_apply.iterrows():
        print(f"  [{row['Category']}] {row['Rule Name']}")
        print(f"        -> {row['Object Name']} :: {row['Proposed Action']}")
    print("\nReview the list above. Set DRY_RUN = False and re-run this cell to apply.")
else:
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
        for _, row in to_apply.iterrows():
            rule = row["Rule Name"]
            try:
                detail = REMEDIATIONS[rule]["fix"](tom, row)
                results.append({**row.to_dict(), "Status": "Applied", "Detail": detail})
            except Exception as e:
                results.append({**row.to_dict(), "Status": "Failed", "Detail": str(e)})
    results_df = pd.DataFrame(results)[["Status", "Category", "Rule Name",
                                        "Object Type", "Object Name", "Detail"]]
    applied = int((results_df["Status"] == "Applied").sum())
    failed = int((results_df["Status"] == "Failed").sum())
    print(f"Done. Applied: {applied} | Failed: {failed}")
    display(results_df)

## Step 5 - Verify

Re-run the BPA. The findings you fixed should be gone. Anything left is either something you chose
to skip, or a rule with no automated fix (manual review).

In [ ]:
labs.run_model_bpa(dataset=dataset, workspace=workspace)

## Notes and caveats

- **Dry run first.** The apply cell writes nothing until `DRY_RUN = False`.
- **Changing a column to Int64** only changes metadata. For Import models the next refresh will fail
  if the source values are not whole numbers; for Direct Lake the source column must be integer.
  Any such failure is reported per-row as `Failed` and does not stop the other fixes.
- **Format strings** are cosmetic and safe to apply widely. Adjust the strings in `REMEDIATIONS`
  to match your locale if needed.
- **Only rules in `REMEDIATIONS` are ever touched.** Every other finding is left for manual review.

**Some findings are left manual on purpose, this is expected, not a failure:**

- *Needs human judgment* - e.g. **"Model should have a date table"**. It could be automated with
  `tom.mark_as_date_table(...)`, but a person should choose the correct date table and column
  (here, `'Valuation Calendar'[Date]`), so it is deliberately not auto-fixed.
- *Not a model change at all* - e.g. **"Ensure Direct Lake source tables are V-Ordered"**. V-Order is
  a physical property of the **Delta tables in the Lakehouse**, not a model setting, so no model edit
  can clear it. Fix it on the Lakehouse by re-writing / `OPTIMIZE`-ing the source tables with V-Order
  enabled (e.g. Spark `OPTIMIZE <table> VORDER`).
- After a successful apply, the change is committed when the `with connect_semantic_model(...)` block
  exits. Re-run Step 1/Step 5 to confirm.